In [1]:
import pandas as pd
import numpy as np

# Load the raw CSV
df = pd.read_csv("../data/raw/healthcare_fraud_detection.csv")

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
df.head(3)

Loaded 10,000 rows and 20 columns


,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.0
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.0
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.0


In [2]:
# First, create a "data quality flag" BEFORE we fill anything in
# This marks rows that originally had at least one missing value
df["Data_Quality_Flag"] = (
    df["Insurance_Type"].isnull() |
    df["Provider_Specialty"].isnull() |
    df["Prior_Visits_12m"].isnull()
).astype(int)

# Now fill the missing values
df["Insurance_Type"] = df["Insurance_Type"].fillna("Unknown")
df["Provider_Specialty"] = df["Provider_Specialty"].fillna("Unknown")

# For the numeric column, use the median
median_prior_visits = df["Prior_Visits_12m"].median()
df["Prior_Visits_12m"] = df["Prior_Visits_12m"].fillna(median_prior_visits)

# Confirm no missing values remain
print("Missing values after imputation:")
print(df.isnull().sum())
print()
print(f"Data quality flag = 1 in {df['Data_Quality_Flag'].sum():,} rows")
print(f"Median used for Prior_Visits_12m: {median_prior_visits}")

Missing values after imputation:
Provider_ID                              0
Claim_ID                                 0
Patient_Age                              0
Patient_Gender                           0
Diagnosis_Code                           0
Procedure_Code                           0
Claim_Amount                             0
Approved_Amount                          0
Insurance_Type                           0
Claim_Submission_Date                    0
Days_Between_Service_and_Claim           0
Number_of_Claims_Per_Provider_Monthly    0
Provider_Specialty                       0
Patient_State                            0
Claim_Status                             0
Is_Fraud                                 0
Length_of_Stay                           0
Visit_Type                               0
Chronic_Condition_Flag                   0
Prior_Visits_12m                         0
Data_Quality_Flag                        0
dtype: int64

Data quality flag = 1 in 1,011 rows
Median used fo

In [3]:
# Convert the string date to a real datetime
df["Claim_Submission_Date"] = pd.to_datetime(df["Claim_Submission_Date"])

# Now extract useful date parts we'll use in Power BI
df["Submission_Year"] = df["Claim_Submission_Date"].dt.year
df["Submission_Month"] = df["Claim_Submission_Date"].dt.month
df["Submission_MonthName"] = df["Claim_Submission_Date"].dt.strftime("%B")

# Confirm it worked
print("Date type:", df["Claim_Submission_Date"].dtype)
print("Date range:", df["Claim_Submission_Date"].min(), "to", df["Claim_Submission_Date"].max())
print()
print(df[["Claim_Submission_Date", "Submission_Year", "Submission_Month", "Submission_MonthName"]].head())

Date type: datetime64[us]
Date range: 2021-01-06 00:00:00 to 2025-01-26 00:00:00

  Claim_Submission_Date  Submission_Year  Submission_Month  \
0            2024-09-01             2024                 9   
1            2022-09-05             2022                 9   
2            2022-04-11             2022                 4   
3            2023-10-11             2023                10   
4            2023-09-05             2023                 9   

  Submission_MonthName  
0            September  
1            September  
2                April  
3              October  
4            September  


In [4]:
# Denied_Flag: 1 if claim was rejected, 0 otherwise
df["Denied_Flag"] = (df["Claim_Status"] == "Rejected").astype(int)

# Denied_Amount: how much of the claim was NOT paid
df["Denied_Amount"] = df["Claim_Amount"] - df["Approved_Amount"]

# Payment_Ratio: what fraction of the claim was actually paid (0.0 to 1.0)
# Guard against divide-by-zero using np.where
df["Payment_Ratio"] = np.where(
    df["Claim_Amount"] > 0,
    df["Approved_Amount"] / df["Claim_Amount"],
    0
)

# Sanity check
print(f"Denied claims: {df['Denied_Flag'].sum():,} ({df['Denied_Flag'].mean():.2%})")
print(f"Total claim amount: ${df['Claim_Amount'].sum():,.2f}")
print(f"Total approved amount: ${df['Approved_Amount'].sum():,.2f}")
print(f"Total denied amount: ${df['Denied_Amount'].sum():,.2f}")
print(f"Average payment ratio: {df['Payment_Ratio'].mean():.2%}")

Denied claims: 1,748 (17.48%)
Total claim amount: $5,728,044.06
Total approved amount: $4,755,141.57
Total denied amount: $972,902.49
Average payment ratio: 84.88%


In [7]:
# Compute percentile-based thresholds from the actual data
# We use the 75th percentile — flagging claims in the top 25% of each metric

late_filing_threshold = df["Days_Between_Service_and_Claim"].quantile(0.75)
long_stay_threshold = df["Length_of_Stay"].quantile(0.75)
high_utilization_threshold = df["Prior_Visits_12m"].quantile(0.75)

print(f"Late filing threshold (75th percentile): {late_filing_threshold} days")
print(f"Long stay threshold (75th percentile): {long_stay_threshold} days")
print(f"High utilization threshold (75th percentile): {high_utilization_threshold} visits")
print()


def assign_denial_reason(row):
    # Rule 1: Fraud always wins
    if row["Is_Fraud"] == 1:
        return "Suspected Fraud"
    
    # Rule 2: Late filing — top 25% of submission delays
    if row["Days_Between_Service_and_Claim"] > late_filing_threshold:
        return "Late Filing"
    
    # Rule 3: Long stays — top 25% of stay durations
    if row["Length_of_Stay"] > long_stay_threshold:
        return "Extended Stay Review"
    
    # Rule 4: Over-utilization — top 25% of prior visits
    if row["Prior_Visits_12m"] > high_utilization_threshold:
        return "Excessive Utilization"
    
    # Rule 5: Everything else
    return "Not Medically Necessary / Policy Exclusion"


# Apply only to denied claims
df["Denial_Reason"] = np.where(
    df["Claim_Status"] == "Rejected",
    df.apply(assign_denial_reason, axis=1),
    "N/A"
)

# Show the distribution
print("Denial reasons for the 1,748 denied claims:")
print(df[df["Claim_Status"] == "Rejected"]["Denial_Reason"].value_counts())
print()
print("As percentages:")
print(df[df["Claim_Status"] == "Rejected"]["Denial_Reason"].value_counts(normalize=True).apply(lambda x: f"{x:.2%}"))

Late filing threshold (75th percentile): 22.0 days
Long stay threshold (75th percentile): 3.0 days
High utilization threshold (75th percentile): 4.0 visits

Denial reasons for the 1,748 denied claims:
Denial_Reason
Not Medically Necessary / Policy Exclusion    671
Suspected Fraud                               404
Late Filing                                   327
Extended Stay Review                          201
Excessive Utilization                         145
Name: count, dtype: int64

As percentages:
Denial_Reason
Not Medically Necessary / Policy Exclusion    38.39%
Suspected Fraud                               23.11%
Late Filing                                   18.71%
Extended Stay Review                          11.50%
Excessive Utilization                          8.30%
Name: proportion, dtype: str


In [9]:
# Look at the actual distributions of the columns used in our rules
print("Days_Between_Service_and_Claim:")
print(df["Days_Between_Service_and_Claim"].describe())
print()
print("Length_of_Stay:")
print(df["Length_of_Stay"].describe())
print()
print("Prior_Visits_12m:")
print(df["Prior_Visits_12m"].describe())

Days_Between_Service_and_Claim:
count    10000.000000
mean        14.413800
std          8.489875
min          0.000000
25%          7.000000
50%         14.000000
75%         22.000000
max         29.000000
Name: Days_Between_Service_and_Claim, dtype: float64

Length_of_Stay:
count    10000.00000
mean         2.19930
std          1.71046
min          0.00000
25%          1.00000
50%          2.00000
75%          3.00000
max          5.00000
Name: Length_of_Stay, dtype: float64

Prior_Visits_12m:
count    10000.000000
mean         3.025500
std          1.692376
min          0.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         12.000000
Name: Prior_Visits_12m, dtype: float64


In [12]:
# Age groups — 5 buckets, with 65+ specifically split out because it's Medicare-eligible
def bucket_age(age):
    if age < 18:
        return "0-17"
    elif age < 35:
        return "18-34"
    elif age < 55:
        return "35-54"
    elif age < 65:
        return "55-64"
    else:
        return "65+"

df["Age_Group"] = df["Patient_Age"].apply(bucket_age)


# Claim value buckets — 4 tiers based on billed amount
def bucket_claim_value(amount):
    if amount < 500:
        return "Small (<$500)"
    elif amount < 2000:
        return "Medium ($500-2K)"
    elif amount < 5000:
        return "Large ($2K-5K)"
    else:
        return "Very Large ($5K+)"

df["Claim_Value_Bucket"] = df["Claim_Amount"].apply(bucket_claim_value)


# Show the distributions
print("Age Group distribution:")
print(df["Age_Group"].value_counts().sort_index())
print()
print("Claim Value Bucket distribution:")
print(df["Claim_Value_Bucket"].value_counts())
print()

# Show denial rate by each bucket — a quick preview of what your dashboard will reveal
print("Denial rate by Age Group:")
print(df.groupby("Age_Group")["Denied_Flag"].mean().apply(lambda x: f"{x:.2%}"))
print()
print("Denial rate by Claim Value Bucket:")
print(df.groupby("Claim_Value_Bucket")["Denied_Flag"].mean().apply(lambda x: f"{x:.2%}"))

Age Group distribution:
Age_Group
0-17      340
18-34    1646
35-54    4083
55-64    1850
65+      2081
Name: count, dtype: int64

Claim Value Bucket distribution:
Claim_Value_Bucket
Small (<$500)        5505
Medium ($500-2K)     4366
Large ($2K-5K)        128
Very Large ($5K+)       1
Name: count, dtype: int64

Denial rate by Age Group:
Age_Group
0-17     17.06%
18-34    17.98%
35-54    17.63%
55-64    17.30%
65+      17.01%
Name: Denied_Flag, dtype: str

Denial rate by Claim Value Bucket:
Claim_Value_Bucket
Large ($2K-5K)       27.34%
Medium ($500-2K)     18.80%
Small (<$500)        16.20%
Very Large ($5K+)     0.00%
Name: Denied_Flag, dtype: str


In [13]:
print(df["Claim_Amount"].describe())

count    10000.000000
mean       572.804406
std        406.202437
min         60.210000
25%        305.205000
50%        461.225000
75%        711.365000
max       6590.700000
Name: Claim_Amount, dtype: float64


In [14]:
# Redefine claim value buckets based on the actual data distribution
def bucket_claim_value(amount):
    if amount < 300:
        return "Small (<$300)"
    elif amount < 500:
        return "Medium ($300-500)"
    elif amount < 1000:
        return "Large ($500-1K)"
    else:
        return "Very Large ($1K+)"

# Reapply to the column
df["Claim_Value_Bucket"] = df["Claim_Amount"].apply(bucket_claim_value)

# Check the new distribution
print("New Claim Value Bucket distribution:")
print(df["Claim_Value_Bucket"].value_counts())
print()
print("Denial rate by new Claim Value Bucket:")
print(df.groupby("Claim_Value_Bucket")["Denied_Flag"].mean().apply(lambda x: f"{x:.2%}"))

New Claim Value Bucket distribution:
Claim_Value_Bucket
Large ($500-1K)      3344
Medium ($300-500)    3081
Small (<$300)        2424
Very Large ($1K+)    1151
Name: count, dtype: int64

Denial rate by new Claim Value Bucket:
Claim_Value_Bucket
Large ($500-1K)      18.03%
Medium ($300-500)    16.39%
Small (<$300)        15.97%
Very Large ($1K+)    21.98%
Name: Denied_Flag, dtype: str


In [15]:
# Quick sanity check before saving
print(f"Final shape: {len(df):,} rows × {len(df.columns)} columns")
print()
print("Column list:")
for col in df.columns:
    print(f"  - {col}")
print()

# Save to the clean folder
output_path = "../data/clean/claims_clean.csv"
df.to_csv(output_path, index=False)

print(f"✓ Saved cleaned dataset to {output_path}")
print(f"✓ File is ready for Power BI import")

Final shape: 10,000 rows × 30 columns

Column list:
  - Provider_ID
  - Claim_ID
  - Patient_Age
  - Patient_Gender
  - Diagnosis_Code
  - Procedure_Code
  - Claim_Amount
  - Approved_Amount
  - Insurance_Type
  - Claim_Submission_Date
  - Days_Between_Service_and_Claim
  - Number_of_Claims_Per_Provider_Monthly
  - Provider_Specialty
  - Patient_State
  - Claim_Status
  - Is_Fraud
  - Length_of_Stay
  - Visit_Type
  - Chronic_Condition_Flag
  - Prior_Visits_12m
  - Data_Quality_Flag
  - Submission_Year
  - Submission_Month
  - Submission_MonthName
  - Denied_Flag
  - Denied_Amount
  - Payment_Ratio
  - Denial_Reason
  - Age_Group
  - Claim_Value_Bucket

✓ Saved cleaned dataset to ../data/clean/claims_clean.csv
✓ File is ready for Power BI import
